#Instalasi Library & Load Dataset

In [ ]:
# Install library datasets dari Hugging Face (jika belum terpasang di Colab)
!pip install -q datasets pandas numpy matplotlib seaborn

from datasets import load_dataset
import pandas as pd
import numpy as np
import re

# Load dataset langsung dari Hugging Face
dataset = load_dataset("AIbnuHibban/e-commerce-sentiment-bahasa-indonesia")

# Konversi split 'train' ke Pandas DataFrame
df = pd.DataFrame(dataset['train'])

print("=== INFORMASI UMUM DATASET ===")
print(f"Jumlah Baris : {df.shape[0]}")
print(f"Jumlah Kolom : {df.shape[1]}")
print("\n5 Data Teratas:")
display(df.head())

README.md:   0%|          | 0.00/2.31k [00:00<?, ?B/s]

challange.json:   0%|          | 0.00/634k [00:00<?, ?B/s]

simple.json:   0%|          | 0.00/1.86M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21840 [00:00<?, ? examples/s]

=== INFORMASI UMUM DATASET ===
Jumlah Baris : 21840
Jumlah Kolom : 4

5 Data Teratas:


,comment,sentiment,category,rating
0,Wah mantap banget nih ditipu! Terima kasih sel...,negative,sarcasm,1
1,"Bagus sekali, barang rusak semua! Top markotop!",negative,sarcasm,1
2,Pelayanan terbaik! Nungguin 2 minggu cuma buat...,negative,sarcasm,1
3,Seneng banget barangnya cacat. Recommended sel...,negative,sarcasm,1
4,Keren nih seller! Kirim barang yang salah deng...,negative,sarcasm,1


##Pemeriksaan Kondisi Data (Untuk Mengisi Poin 4.3)

In [ ]:
print("=========================================")
print("1. PEMERIKSAAN MISSING VALUES")
print("=========================================")
missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Jumlah Missing': missing_count, 'Persentase (%)': missing_percent.round(2)})
print(missing_df)

print("\n=========================================")
print("2. PEMERIKSAAN DUPLICATE DATA")
print("=========================================")
dup_all = df.duplicated().sum()
dup_comment = df.duplicated(subset=['comment']).sum()
print(f"Duplikat seluruh kolom     : {dup_all} baris ({(dup_all/len(df)*100):.2f}%)")
print(f"Duplikat pada teks comment : {dup_comment} baris ({(dup_comment/len(df)*100):.2f}%)")

print("\n=========================================")
print("3. PEMERIKSAAN OUTLIERS (PANJANG TEKS & RATING)")
print("=========================================")
df['char_length'] = df['comment'].astype(str).apply(len)
df['word_count'] = df['comment'].astype(str).apply(lambda x: len(x.split()))

print("Statistik Panjang Karakter:")
print(df['char_length'].describe())
print("\nStatistik Jumlah Kata:")
print(df['word_count'].describe())

# Cek outlier dengan metode IQR pada panjang karakter
Q1 = df['char_length'].quantile(0.25)
Q3 = df['char_length'].quantile(0.75)
IQR = Q3 - Q1
outliers_text = df[(df['char_length'] < (Q1 - 1.5 * IQR)) | (df['char_length'] > (Q3 + 1.5 * IQR))]
print(f"\nJumlah Outlier Panjang Teks (Metode IQR): {len(outliers_text)} baris ({(len(outliers_text)/len(df)*100):.2f}%)")
print("Distribusi Nilai Rating:")
print(df['rating'].value_counts().sort_index())

print("\n=========================================")
print("4. PEMERIKSAAN IMBALANCED DATA (DISTRIBUSI KELAS)")
print("=========================================")
sent_counts = df['sentiment'].value_counts()
sent_percent = df['sentiment'].value_counts(normalize=True) * 100
imb_df = pd.DataFrame({'Jumlah': sent_counts, 'Persentase (%)': sent_percent.round(2)})
print(imb_df)

print("\nDistribusi Kategori/Aspek (Top 15):")
print(df['category'].value_counts())

print("\n=========================================")
print("5. PEMERIKSAAN NOISE & DATA QUALITY ISSUES")
print("=========================================")
# Cek teks kosong atau hanya spasi
empty_text = df[df['comment'].astype(str).str.strip() == ''].shape[0]
# Cek mengandung angka
has_digits = df['comment'].astype(str).apply(lambda x: bool(re.search(r'\d', x))).sum()
# Cek mengandung karakter non-ASCII (emoji / simbol khusus)
has_emoji_non_ascii = df['comment'].astype(str).apply(lambda x: bool(re.search(r'[^\x00-\x7F]', x))).sum()
# Cek mengandung tanda baca berulang (contoh: !!! atau ???)
has_repeated_punct = df['comment'].astype(str).apply(lambda x: bool(re.search(r'[!?.,]{2,}', x))).sum()

print(f"Teks kosong / hanya spasi      : {empty_text} baris")
print(f"Mengandung angka               : {has_digits} baris ({(has_digits/len(df)*100):.2f}%)")
print(f"Mengandung emoji / non-ASCII   : {has_emoji_non_ascii} baris ({(has_emoji_non_ascii/len(df)*100):.2f}%)")
print(f"Mengandung tanda baca berulang : {has_repeated_punct} baris ({(has_repeated_punct/len(df)*100):.2f}%)")

1. PEMERIKSAAN MISSING VALUES
           Jumlah Missing  Persentase (%)
comment                 0            0.00
sentiment               0            0.00
category            17000           77.84
rating                  0            0.00

2. PEMERIKSAAN DUPLICATE DATA
Duplikat seluruh kolom     : 14040 baris (64.29%)
Duplikat pada teks comment : 14815 baris (67.83%)

3. PEMERIKSAAN OUTLIERS (PANJANG TEKS & RATING)
Statistik Panjang Karakter:
count    21840.000000
mean        34.498764
std         13.954506
min          3.000000
25%         25.000000
50%         33.000000
75%         45.000000
max         78.000000
Name: char_length, dtype: float64

Statistik Jumlah Kata:
count    21840.000000
mean         5.165842
std          2.101279
min          1.000000
25%          3.000000
50%          5.000000
75%          7.000000
max         13.000000
Name: word_count, dtype: float64

Jumlah Outlier Panjang Teks (Metode IQR): 3 baris (0.01%)
Distribusi Nilai Rating:
rating
1    5325
2    216

##Data Quality Issues & Potensi Data Leakage

In [ ]:
from sklearn.model_selection import train_test_split

print("==========================================================")
print("A. DATA QUALITY ISSUES (KESESUAIAN LABEL & TEKS BERMASALAH)")
print("==========================================================")

# 1. Cek inkonsistensi antara Rating dan Sentiment (contoh: rating 5 tapi sentimen negative, atau rating 1 tapi positive)
inconsistent_pos = df[(df['rating'] <= 2) & (df['sentiment'] == 'positive')]
inconsistent_neg = df[(df['rating'] >= 4) & (df['sentiment'] == 'negative')]

print(f"Rating rendah (1-2) tapi label 'positive' : {len(inconsistent_pos)} baris")
print(f"Rating tinggi (4-5) tapi label 'negative' : {len(inconsistent_neg)} baris (termasuk kategori sarcasm)")

# Cek berapa banyak dari inkonsistensi tersebut yang memang berlabel kategori 'sarcasm'
if len(inconsistent_neg) > 0:
    sarcasm_count = inconsistent_neg[inconsistent_neg['category'] == 'sarcasm'].shape[0]
    print(f"  -> Dari jumlah tersebut, yang termasuk kategori 'sarcasm': {sarcasm_count} baris")

# 2. Cek komentar yang sangat pendek (< 5 karakter) yang berpotensi minim informasi
very_short = df[df['comment'].astype(str).str.strip().str.len() < 5]
print(f"Komentar terlalu pendek (< 5 karakter)    : {len(very_short)} baris")


print("\n==========================================================")
print("B. ASUMSI DATA, DISTRIBUSI TRAIN-TEST & POTENSI DATA LEAKAGE")
print("==========================================================")

# Bersihkan duplikat terlebih dahulu sebelum split agar tidak terjadi data leakage
df_clean = df.drop_duplicates(subset=['comment']).copy()

X = df_clean['comment']
y = df_clean['sentiment']

# Stratified Split 80% Train, 20% Test (memastikan asumsi i.i.d. dan distribusi proporsional)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Cek potensi Data Leakage: apakah ada teks ulasan yang bocor (sama persis) di Train dan Test
overlap_count = len(set(X_train).intersection(set(X_test)))

print(f"Total data bersih (setelah deduplikasi)   : {len(df_clean)} baris")
print(f"Data Latih (Train 80%)                    : {len(X_train)} baris")
print(f"Data Uji (Test 20%)                       : {len(X_test)} baris")
print(f"Irisan teks identik (Data Leakage Check)  : {overlap_count} baris")
print("\nPerbandingan Distribusi Kelas (%):")
dist_df = pd.DataFrame({
    'Train (%)': (y_train.value_counts(normalize=True) * 100).round(2),
    'Test (%)': (y_test.value_counts(normalize=True) * 100).round(2)
})
print(dist_df)

A. DATA QUALITY ISSUES (KESESUAIAN LABEL & TEKS BERMASALAH)
Rating rendah (1-2) tapi label 'positive' : 0 baris
Rating tinggi (4-5) tapi label 'negative' : 0 baris (termasuk kategori sarcasm)
Komentar terlalu pendek (< 5 karakter)    : 7 baris

B. ASUMSI DATA, DISTRIBUSI TRAIN-TEST & POTENSI DATA LEAKAGE
Total data bersih (setelah deduplikasi)   : 7025 baris
Data Latih (Train 80%)                    : 5620 baris
Data Uji (Test 20%)                       : 1405 baris
Irisan teks identik (Data Leakage Check)  : 0 baris

Perbandingan Distribusi Kelas (%):
           Train (%)  Test (%)
sentiment                     
positive       40.48     40.50
negative       39.98     39.93
neutral        19.54     19.57


In [ ]:
pos_pct = sent_percent.get('positive', 0).round(2)
neu_pct = sent_percent.get('neutral', 0).round(2)
neg_pct = sent_percent.get('negative', 0).round(2)

print(f"""4.3 Kondisi Data
- Missing Values: Tidak ditemukan nilai kosong (missing values) pada seluruh kolom dataset (0% pada kolom comment, sentiment, category, dan rating), sebagaimana telah divalidasi secara menyeluruh menggunakan df.isnull().sum() sebelum tahap pemrosesan.
- Duplicate Data: Ditemukan sebanyak {dup_comment} baris ({(dup_comment/len(df)*100):.2f}%) duplikasi teks pada kolom comment (dan {dup_all} duplikat identik di seluruh kolom). Data duplikat tersebut dihapus menggunakan drop_duplicates(subset=['comment']) sebelum pembagian data agar tidak memicu bias saat pelatihan model.
- Outliers: Pada fitur numerik rating, tidak terdapat outlier karena nilai berada dalam rentang skala tetap 1 hingga 5. Pada fitur teks comment, panjang ulasan berkisar antara {df['char_length'].min()} hingga {df['char_length'].max()} karakter, dengan terdeteksi sebanyak {len(outliers_text)} baris ({(len(outliers_text)/len(df)*100):.2f}%) outlier panjang karakter berdasarkan metode IQR (ulasan yang terlalu singkat atau terlalu panjang).
- Imbalanced Data: Distribusi kelas pada kolom target sentiment tergolong cukup seimbang (balanced) dengan proporsi kelas positive sebesar {pos_pct}%, neutral sebesar {neu_pct}%, dan negative sebesar {neg_pct}%. Ketidakseimbangan ringan hanya terlihat pada distribusi rating yang cenderung mengumpul pada skor ekstrem (1, 4, dan 5).
- Noise: Ditemukan elemen noise khas ulasan e-commerce berbahasa Indonesia, meliputi penggunaan bahasa gaul/tidak baku, singkatan ("ga", "nih"), angka, tanda baca berulang, serta emoji (seperti 😊 dan 🙏). Selain itu, terdapat kategori khusus sarkasme (sarcasm) (misalnya kalimat pujian yang bermakna sindiran negatif) yang menjadi tantangan utama dalam pemrosesan teks.
- Data Quality Issues: Terdapat ketidaksesuaian permukaan antara skor rating dan label sentiment (sebanyak {len(inconsistent_neg)} baris memiliki rating tinggi 4–5 namun berlabel negative), yang setelah diperiksa merupakan ulasan berkategori sarcasm. Selain itu, ditemukan {len(very_short)} ulasan yang terlalu pendek (< 5 karakter) serta teks duplikat yang dibersihkan pada tahap preprocessing.
- Asumsi Data (i.i.d., distribusi train-test, potensi data leakage): Data diasumsikan bersifat independent and identically distributed (i.i.d.). Untuk mencegah potensi data leakage, dilakukan penghapusan data duplikat terlebih dahulu sebelum pembagian data menggunakan metode Stratified Train-Test Split (rasio 80:20) sehingga proporsi kelas di train set dan test set tetap identik dengan irisan teks duplikat sebesar {overlap_count} baris, serta proses fitting ekstraksi fitur (TF-IDF) hanya dilakukan pada data latih (training set).""")

4.3 Kondisi Data
- Missing Values: Tidak ditemukan nilai kosong (missing values) pada seluruh kolom dataset (0% pada kolom comment, sentiment, category, dan rating), sebagaimana telah divalidasi secara menyeluruh menggunakan df.isnull().sum() sebelum tahap pemrosesan.
- Duplicate Data: Ditemukan sebanyak 14815 baris (67.83%) duplikasi teks pada kolom comment (dan 14040 duplikat identik di seluruh kolom). Data duplikat tersebut dihapus menggunakan drop_duplicates(subset=['comment']) sebelum pembagian data agar tidak memicu bias saat pelatihan model.
- Outliers: Pada fitur numerik rating, tidak terdapat outlier karena nilai berada dalam rentang skala tetap 1 hingga 5. Pada fitur teks comment, panjang ulasan berkisar antara 3 hingga 78 karakter, dengan terdeteksi sebanyak 3 baris (0.01%) outlier panjang karakter berdasarkan metode IQR (ulasan yang terlalu singkat atau terlalu panjang).
- Imbalanced Data: Distribusi kelas pada kolom target sentiment tergolong cukup seimbang (balanced) deng